### Step 1: Importing Libraries

In [ ]:
# Importing the libraries
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

### Step 2: Download and Extract Datasets

In [ ]:
# Download MovieLens 100K dataset
url_100k = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
urllib.request.urlretrieve(url_100k, "ml-100k.zip")

# Download MovieLens 1M dataset
url_1m = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
urllib.request.urlretrieve(url_1m, "ml-1m.zip")

# Extract the datasets
with zipfile.ZipFile("ml-100k.zip", 'r') as zip_ref:
    zip_ref.extractall("ml-100k")

with zipfile.ZipFile("ml-1m.zip", 'r') as zip_ref:
    zip_ref.extractall("ml-1m")

### Step 3: Importing Movies Dataset

In [ ]:
# Importing the dataset
# The movies.dat file is read into a pandas DataFrame.
# The separator is '::', there is no header, the engine is set to 'python' for handling the separator, and the encoding is 'latin-1'.
movies = pd.read_csv('ml-1m/ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
# Display the first 5 rows of the DataFrame.
movies.head()

,0,1,2
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


### Step 4: Importing Users Dataset

In [ ]:
# Importing the users dataset
# The users.dat file is read into a pandas DataFrame with similar settings as the movies dataset.
users = pd.read_csv('ml-1m/ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
# Display the first 5 rows of the DataFrame.
users.head()

,0,1,2,3,4
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


### Step 5: Importing Ratings Dataset

In [ ]:
# Importing the ratings dataset
# The ratings.dat file is read into a pandas DataFrame with similar settings as the movies and users datasets.
ratings = pd.read_csv('ml-1m/ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
# Display the first 5 rows of the DataFrame.
ratings.head()

,0,1,2,3
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


### Step 6: Preparing Training and Test Sets

In [ ]:
# Preparing the training set and the test set
# The u1.base file from the 100k dataset is read into a pandas DataFrame as the training set.
training_set = pd.read_csv('./ml-100k/ml-100k/u1.base', delimiter = '\t')
# The u1.test file from the 100k dataset is read into a pandas DataFrame as the test set.
test_set = pd.read_csv('./ml-100k/ml-100k/u1.test', delimiter = '\t')

# Convert the training set and the test set into numpy arrays
training_set = np.array(training_set, dtype='int')
test_set = np.array(test_set, dtype='int')

### Step 7: Getting Total Number of Users and Movies

In [ ]:
# getting the total number of users and movies
# Find the maximum user ID from both training and test sets to get the total number of users.
nb_users = int(max(max(training_set[:,0]), max(test_set[:,0])))
# Find the maximum movie ID from both training and test sets to get the total number of movies.
nb_movies = int(max(max(training_set[:,1]), max(test_set[:,1])))

# Print the total number of users and movies.
print(nb_users)
print(nb_movies)

943
1682


### Step 8: Converting Data into Array

In [ ]:
# Converting the data into array with users in rows and movies in columns
# This function converts the data (training or test set) into a list of lists,
# where each inner list represents a user and contains their ratings for all movies.
# If a user hasn't rated a movie, the rating is 0.
def convert(data):
    new_data = []
    for id_users in range(1, nb_users + 1):
        # Get the movie IDs rated by the current user.
        id_movies = data[:,1][data[:,0] == id_users]
        # Get the ratings given by the current user.
        id_ratings = data[:,2][data[:,0] == id_users]
        # Create a numpy array of zeros with the size of total number of movies.
        ratings = np.zeros(nb_movies)
        # Fill in the ratings for the movies the user has rated.
        ratings[id_movies - 1] = id_ratings
        # Append the list of ratings for the current user to the new_data list.
        new_data.append(list(ratings))
    return new_data

# Convert the training and test sets using the convert function.
training_set = convert(training_set)
test_set = convert(test_set)

### Step 9: Convert Data into Torch Tensors

In [ ]:
# Convert the data into torch tensors
# Convert the training and test sets from numpy arrays to PyTorch FloatTensors.
training_set = torch.FloatTensor(training_set)
test_set = torch.FloatTensor(test_set)

### Step 10: Install torchinfo

In [ ]:
# Install torchinfo library to get a summary of the model.
# !pip install torchinfo

### Step 11: Creating the Architecture of the Neural Network

In [ ]:
# Creating the architecture of the neural network
# Define a Stacked Autoencoder (SAE) class inheriting from nn.Module.
class SAE(nn.Module):
    def __init__(self):
        super(SAE, self).__init__()
        # First linear layer: input size nb_movies, output size 20 (encoder)
        self.fc1 = nn.Linear(nb_movies, 20)
        # Second linear layer: input size 20, output size 10 (encoder)
        self.fc2 = nn.Linear(20, 10)
        # Third linear layer: input size 10, output size 20 (decoder)
        self.fc3 = nn.Linear(10, 20)
        # Fourth linear layer: input size 20, output size nb_movies (decoder)
        self.fc4 = nn.Linear(20, nb_movies)
        # Sigmoid activation function
        self.activation = nn.Sigmoid()

    # Define the forward pass of the network
    def forward(self, x):
        # Apply sigmoid activation after the first linear layer
        x = self.activation(self.fc1(x))
        # Apply sigmoid activation after the second linear layer
        x = self.activation(self.fc2(x))
        # Apply sigmoid activation after the third linear layer
        x = self.activation(self.fc3(x))
        # Apply the fourth linear layer (output layer)
        x = self.fc4(x)
        return x

# Create an instance of the SAE model.
sae = SAE()
# Define the criterion (loss function) as Mean Squared Error.
criterion = nn.MSELoss()

# Define the optimizer as RMSprop with a learning rate of 0.01 and weight decay of 0.5.
optimizer = optim.RMSprop(sae.parameters(), lr = 0.01, weight_decay = 0.5)

# Import summary from torchinfo to display model architecture and parameters.
from torchinfo import summary
# Print a summary of the SAE model with an example input size.
summary(sae, input_size=(1, nb_movies))

Layer (type:depth-idx)                   Output Shape              Param #
SAE                                      [1, 1682]                 --
├─Linear: 1-1                            [1, 20]                   33,660
├─Sigmoid: 1-2                           [1, 20]                   --
├─Linear: 1-3                            [1, 10]                   210
├─Sigmoid: 1-4                           [1, 10]                   --
├─Linear: 1-5                            [1, 20]                   220
├─Sigmoid: 1-6                           [1, 20]                   --
├─Linear: 1-7                            [1, 1682]                 35,322
Total params: 69,412
Trainable params: 69,412
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.07
Input size (MB): 0.01
Forward/backward pass size (MB): 0.01
Params size (MB): 0.28
Estimated Total Size (MB): 0.30

### Step 12: Training the SAE

In [ ]:
# Training the SAE
# Define the number of training epochs.
np_epochs = 200
# Check if GPU is available and set the device accordingly.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Move the model to the selected device.
sae.to(device)

# Loop through each epoch.
for epoch in range(1, np_epochs + 1):
    # Initialize the training loss for the current epoch.
    train_loss = 0
    # Initialize a counter for the number of users with ratings.
    s = 0.
    # Loop through each user in the training set.
    for id_user in range(nb_users):
        # Get the input data for the current user and add a batch dimension.
        # Move the input tensor to the selected device.
        input = Variable(training_set[id_user]).unsqueeze(0).to(device)
        # Create a target variable by cloning the input.
        # Move the target tensor to the selected device.
        target = input.clone().to(device)
        # Check if the current user has rated any movies.
        if torch.sum(target.data > 0) > 0:
            # Pass the input through the SAE model to get the output (predicted ratings).
            output = sae(input)
            # Set the predicted ratings to 0 for movies the user hasn't rated (masking).
            output[target == 0] = 0
            # Calculate the loss between the output and the target.
            loss = criterion(output, target)
            # Calculate a mean corrector to account for the number of rated movies.
            mean_corrector = nb_movies / float(torch.sum(target.data > 0) + 1e-10)
            # Perform backpropagation to calculate gradients.
            loss.backward()
            # Accumulate the training loss (using RMSE).
            train_loss += np.sqrt(loss.data * mean_corrector)
            # Increment the counter for users with ratings.
            s += 1.
            # Perform a single optimization step to update model parameters.
            optimizer.step()
    # Print the average training loss for the current epoch.
    print('epoch: {}, loss: {}'.format(epoch, train_loss / s))

/tmp/ipython-input-3682096450.py:36: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  train_loss += np.sqrt(loss.data * mean_corrector)


epoch: 1, loss: 1.7720494270324707
epoch: 2, loss: 1.0967594385147095
epoch: 3, loss: 1.0536373853683472
epoch: 4, loss: 1.0382471084594727
epoch: 5, loss: 1.0309467315673828
epoch: 6, loss: 1.0266104936599731
epoch: 7, loss: 1.0237730741500854
epoch: 8, loss: 1.0220789909362793
epoch: 9, loss: 1.0207269191741943
epoch: 10, loss: 1.019632339477539
epoch: 11, loss: 1.0188826322555542
epoch: 12, loss: 1.0185693502426147
epoch: 13, loss: 1.0179736614227295
epoch: 14, loss: 1.017670750617981
epoch: 15, loss: 1.0171018838882446
epoch: 16, loss: 1.0168007612228394
epoch: 17, loss: 1.0168981552124023
epoch: 18, loss: 1.016417145729065
epoch: 19, loss: 1.016494870185852
epoch: 20, loss: 1.0160564184188843
epoch: 21, loss: 1.0160040855407715
epoch: 22, loss: 1.0157333612442017
epoch: 23, loss: 1.0157618522644043
epoch: 24, loss: 1.0156346559524536
epoch: 25, loss: 1.0156230926513672
epoch: 26, loss: 1.015268087387085
epoch: 27, loss: 1.01536226272583
epoch: 28, loss: 1.0149787664413452
epoch: 2

### Step 13: Evaluating the SAE on the Test Set

In [ ]:
# Evaluating the SAE on the test set
# Initialize the test loss.
test_loss = 0
# Initialize a counter for the number of users with ratings in the test set.
s = 0
# Loop through each user.
for id_user in range(nb_users):
    # Get the input data for the current user from the training set and add a batch dimension.
    # We use the training set as input to predict ratings for the test set.
    input = Variable(training_set[id_user]).unsqueeze(0)
    # Get the target data for the current user from the test set and add a batch dimension.
    target = Variable(test_set[id_user]).unsqueeze(0)
    # Check if the current user has rated any movies in the test set.
    if torch.sum(target.data > 0) > 0:
        # Pass the input through the trained SAE model to get the output (predicted ratings).
        output = sae(input)
        # Set the predicted ratings to 0 for movies the user hasn't rated in the test set (masking).
        output[target == 0] = 0  # Mask out unrated movies
        # Calculate the loss between the output and the target for the test set.
        loss = criterion(output, target)
        # Calculate a mean corrector to account for the number of rated movies in the test set.
        mean_corrector = nb_movies / float(torch.sum(target.data > 0) + 1e-10)
        # Accumulate the test loss (using RMSE).
        test_loss += np.sqrt(loss.data * mean_corrector)
        # Increment the counter for users with ratings in the test set.
        s += 1.
# Print the average test loss.
print('test loss: {}'.format(test_loss / s))

/tmp/ipython-input-1207870831.py:24: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  test_loss += np.sqrt(loss.data * mean_corrector)


test loss: 0.9510493278503418


### Step 14: Make Predictions and Process Them

In [ ]:
# Make predictions and process them
all_predictions = []
for id_user in range(nb_users):
    # Get the input data for the current user from the training set.
    input = Variable(training_set[id_user]).unsqueeze(0)
    # Get the target data for the current user from the test set.
    target = Variable(test_set[id_user]).unsqueeze(0)

    # Pass the input through the trained SAE model to get the output (predicted ratings).
    output = sae(input)

    # Clone the output to store the processed predictions.
    processed_predictions = output.clone()

    # Where the user has rated the movie in the test set, use the actual rating.
    # Where the user has not rated the movie (rating is 0), keep the predicted rating.
    processed_predictions[target > 0] = target[target > 0]

    # Append the processed predictions for the current user.
    all_predictions.append(processed_predictions.squeeze(0).detach().numpy())

# Convert the list of predictions to a numpy array for easier handling.
all_predictions = np.array(all_predictions)

### Step 15: Calculating the RMSE

In [ ]:
# Calculating the RMSE using the raw model output
test_loss = 0
s = 0.
sae.eval() # Set the model to evaluation mode
with torch.no_grad(): # Disable gradient calculation for evaluation
    for id_user in range(nb_users):
        # Get the actual ratings from the test set for the current user.
        actual_ratings = test_set[id_user]
        # Get the training ratings for the current user.
        training_ratings = training_set[id_user]

        # Get the raw model output for the current user from the training set input.
        # Move input to the correct device if using GPU
        input = Variable(training_set[id_user]).unsqueeze(0).to(device)
        raw_predictions = sae(input)

        # Create a mask for movies rated in the test set and not in the training set
        rated_in_test_mask = actual_ratings > 0
        not_rated_in_training_mask = training_ratings == 0
        evaluation_mask = rated_in_test_mask & not_rated_in_training_mask

        if torch.sum(evaluation_mask.int()) > 0:
            # Extract raw predicted and actual ratings for evaluation
            predicted_eval = raw_predictions[0][evaluation_mask] # Get the predictions for the single user in the batch
            actual_eval = actual_ratings[evaluation_mask]

            # Calculate the Mean Squared Error (MSE) between predicted and actual ratings.
            loss = torch.nn.functional.mse_loss(predicted_eval, actual_eval)

            # Add the calculated MSE to test_loss.
            test_loss += loss.item()
            # Increment the counter s.
            s += 1.

if s > 0:
    rmse = np.sqrt(test_loss / s)
    print('RMSE (using raw model output): {}'.format(rmse))
else:
    print('No movies found to evaluate RMSE on (rated in test but not in training).')

Corrected RMSE (using raw model output): 0.9839028325890872


### Step 16: Implement Recommendation Function

In [ ]:
def recommend_movies(user_id, all_predictions, movies_df, num_recommendations=10):
    """
    Recommends movies for a given user based on predicted ratings.

    Args:
        user_id (int): The ID of the user for whom to recommend movies.
        all_predictions (np.ndarray): Array of predicted ratings for all users and movies.
        movies_df (pd.DataFrame): DataFrame containing movie information.
        num_recommendations (int): The number of movies to recommend.

    Returns:
        pd.DataFrame: A DataFrame containing the recommended movies and their predicted ratings.
    """
    # Get the predicted ratings for the specified user.
    user_predictions = all_predictions[user_id - 1] # Adjust for 0-based indexing

    # Get the movies the user has already rated from the training set.
    # This assumes the training set contains all user ratings used for training.
    rated_movie_indices = training_set[user_id - 1].nonzero().squeeze()

    # Get the movie indices sorted by predicted rating in descending order.
    sorted_movie_indices = np.argsort(user_predictions)[::-1]

    # Filter out movies the user has already rated.
    recommended_movie_indices = [
        idx for idx in sorted_movie_indices if idx not in rated_movie_indices
    ]

    # Get the top N recommended movie indices.
    top_recommendation_indices = recommended_movie_indices[:num_recommendations]

    # Get the movie IDs from the original movies DataFrame.
    # Add 1 to the index to get the actual movie ID.
    recommended_movie_ids = movies_df.iloc[top_recommendation_indices][0].values

    # Get the predicted ratings for the recommended movies.
    recommended_movie_ratings = user_predictions[top_recommendation_indices]

    # Create a DataFrame of recommendations.
    recommendations = pd.DataFrame({
        'Movie ID': recommended_movie_ids,
        'Predicted Rating': recommended_movie_ratings
    })

    # Merge with the movies DataFrame to get movie titles and genres.
    recommendations = recommendations.merge(movies_df, left_on='Movie ID', right_on=0, how='left')

    # Select relevant columns and rename them.
    recommendations = recommendations[['Movie ID', 1, 2, 'Predicted Rating']]
    recommendations.columns = ['Movie ID', 'Title', 'Genres', 'Predicted Rating']


    return recommendations

### Step 17: Display Recommendations

In [ ]:
# Display recommendations for a sample user (e.g., user with ID 1)
user_id_to_recommend = 1
recommendations = recommend_movies(user_id_to_recommend, all_predictions, movies)

print(f"Top 10 movie recommendations for User {user_id_to_recommend}:")
display(recommendations)

Top 10 movie recommendations for User 1:


,Movie ID,Title,Genres,Predicted Rating
0,1537,Shall We Dance? (Shall We Dansu?) (1996),Comedy,5.212413
1,1498,Inventing the Abbotts (1997),Drama|Romance,5.175505
2,861,Supercop (1992),Action|Thriller,5.024384
3,1476,Private Parts (1997),Comedy|Drama,5.014056
4,81,Things to Do in Denver when You're Dead (1995),Crime|Drama|Romance,5.000000
5,109,Headless Body in Topless Bar (1995),Comedy,5.000000
6,114,Margaret's Museum (1995),Drama,5.000000
7,115,Happiness Is in the Field (1995),Comedy,5.000000
8,97,"Hate (Haine, La) (1995)",Drama,5.000000
9,14,Nixon (1995),Drama,5.000000
